# 03 - Algorithm Analysis & Two-Layer Clustering

Este notebook analiza **TODO el universo de algoritmos** (~14,761) usando un enfoque de clustering de dos capas.

## Enfoque de Dos Capas

**Capa 1: Life Profile (Perfil de Vida)**
- ¿Cuándo empieza el algoritmo?
- ¿Cuánto dura?
- ¿Tiene gaps de actividad?

**Capa 2: Financial Behavior (Comportamiento Financiero)**
- Dentro de cada grupo de vida, ¿cómo se comporta?
- ¿Es rentable? ¿Volátil? ¿Estable?
- ¿Mejora o empeora con el tiempo?

## Por qué dos capas?
No tiene sentido comparar directamente un algoritmo que vive todo el estudio con otro que solo dura 3 semanas. Son contextos diferentes.

## Features Mejoradas

| Bloque | Features | Descripción |
|--------|----------|-------------|
| Actividad | start_idx, duration_ratio, active_ratio, n_gaps | Patrón temporal de vida |
| Rendimiento | sharpe, sortino, max_dd, ulcer_index, hit_ratio | Performance financiera |
| Transición | return_decay, sharpe_decay, time_to_first_dd | Evolución con la edad |
| Benchmark | corr_benchmark, beta, alpha, tracking_error | Relación con benchmark |

## Output
- `data/processed/algo_features_enhanced.parquet`: Features completas
- `data/processed/algo_clusters_twolayer.parquet`: Clusters de dos capas
- `data/processed/clustering_methods_comparison.csv`: Comparación de métodos

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy import stats

from src.data.loader import DataLoader
from src.data.preprocessor import DataPreprocessor
from src.analysis.algo_features import (
    AlgoFeatureExtractor,
    AlgoFeatureConfig,
    ACTIVITY_FEATURES,
    PERFORMANCE_FEATURES,
    TRANSITION_FEATURES,
    BENCHMARK_FEATURES,
)
from src.analysis.algo_clusterer import (
    AlgoClusterer,
    ClusterMethod,
    ScalerType,
    name_clusters,
)
from src.utils.logging_utils import setup_logging

setup_logging(level='INFO')

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)

## 1. Cargar Datos

In [ ]:
from src.utils.paths import data_paths, output_paths
dp = data_paths()
op = output_paths()

# Cargar datos
loader = DataLoader(str(dp.raw.root))
preprocessor = DataPreprocessor(initial_capital=100, resample_freq='D')

# Benchmark
benchmark_raw = loader.load_benchmark()
benchmark = preprocessor.process_benchmark(benchmark_raw)

# Lista de algoritmos en el benchmark
benchmark_algo_ids = set(benchmark.products)
print(f"Algoritmos en benchmark: {len(benchmark_algo_ids)}")

# Cargar TODOS los algoritmos (no solo los del benchmark)
print("\nCargando TODOS los algoritmos...")
all_algo_ids = loader.list_algorithms()
print(f"Total algoritmos disponibles: {len(all_algo_ids)}")

# Cargar en batches para evitar problemas de memoria
all_algorithms = loader.load_all_algorithms(algo_ids=all_algo_ids, show_progress=True)
processed_all_algos = preprocessor.process_all_algorithms(all_algorithms, show_progress=True)

# Matriz de retornos de TODOS los algoritmos
returns_matrix_all = preprocessor.create_returns_matrix(processed_all_algos)

# También necesitamos los del benchmark para calcular benchmark returns
processed_benchmark_algos = {k: v for k, v in processed_all_algos.items() if k in benchmark_algo_ids}
returns_matrix_benchmark = preprocessor.create_returns_matrix(processed_benchmark_algos)

benchmark_daily_returns = preprocessor.calculate_benchmark_daily_returns(
    returns_matrix_benchmark, benchmark.weights
)

print(f"\n{'='*60}")
print(f"RESUMEN DE DATOS")
print(f"{'='*60}")
print(f"Total algoritmos procesados: {len(processed_all_algos)}")
print(f"Algoritmos en benchmark: {len(benchmark_algo_ids)}")
print(f"Matriz ALL: {returns_matrix_all.shape}")
print(f"Rango fechas: {returns_matrix_all.index.min()} -> {returns_matrix_all.index.max()}")

---
## 2. Feature Engineering Avanzado
---

Extraemos features en 4 bloques:
1. **Actividad**: Patrón temporal de vida
2. **Rendimiento**: Métricas financieras
3. **Transición**: Evolución con la edad
4. **Benchmark**: Relación con el benchmark

In [ ]:
# Configurar extractor de features
config = AlgoFeatureConfig(
    min_active_days=60,      # Mínimo 60 días activos
    tercile_split=True,      # Dividir vida en tercios
    compute_autocorr=True,   # Calcular autocorrelaciones
    dd_threshold=0.05,       # Umbral para primer drawdown relevante (5%)
)

extractor = AlgoFeatureExtractor(config)

print("Extrayendo features avanzadas...")
print(f"  - Features de actividad: {ACTIVITY_FEATURES}")
print(f"  - Features de rendimiento: {PERFORMANCE_FEATURES}")
print(f"  - Features de transición: {TRANSITION_FEATURES}")
print(f"  - Features vs benchmark: {BENCHMARK_FEATURES}")

# Extraer features
algo_features = extractor.extract_all_features(
    returns_matrix_all,
    benchmark_returns=benchmark_daily_returns
)

# Añadir flag indicando si está en el benchmark
algo_features['in_benchmark'] = algo_features.index.isin(benchmark_algo_ids)

print(f"\n{'='*60}")
print(f"FEATURES EXTRAÍDAS")
print(f"{'='*60}")
print(f"Total algoritmos con suficientes datos: {len(algo_features)}")
print(f"  - En benchmark: {algo_features['in_benchmark'].sum()}")
print(f"  - Fuera de benchmark: {(~algo_features['in_benchmark']).sum()}")
print(f"\nTotal features: {len(algo_features.columns)}")
print(f"Columns: {list(algo_features.columns)}")

In [ ]:
# Vista previa de features
print("\nMuestra de features:")
display(algo_features.head(10))

In [ ]:
# Estadísticas descriptivas por bloque
print("\n" + "="*80)
print("ESTADÍSTICAS POR BLOQUE DE FEATURES")
print("="*80)

blocks = {
    'Actividad': ACTIVITY_FEATURES,
    'Rendimiento': PERFORMANCE_FEATURES,
    'Transición': TRANSITION_FEATURES,
    'Benchmark': BENCHMARK_FEATURES,
}

for block_name, features in blocks.items():
    available = [f for f in features if f in algo_features.columns]
    if available:
        print(f"\n{block_name}:")
        display(algo_features[available].describe().T.style.format('{:.3f}'))

---
## 3. Análisis de Features de Actividad (Life Profile)
---

¿Cómo se distribuyen los patrones de vida de los algoritmos?

In [ ]:
# Visualizar distribución de features de actividad
activity_features = [f for f in ACTIVITY_FEATURES if f in algo_features.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(activity_features[:8]):
    ax = axes[i]
    
    # Separar benchmark vs no benchmark
    in_bench = algo_features[algo_features['in_benchmark']][col]
    not_in_bench = algo_features[~algo_features['in_benchmark']][col]
    
    ax.hist(not_in_bench, bins=30, alpha=0.5, color='gray', label='No benchmark', density=True)
    ax.hist(in_bench, bins=30, alpha=0.7, color='blue', label='Benchmark', density=True)
    
    ax.axvline(not_in_bench.median(), color='gray', linestyle='--', alpha=0.7)
    ax.axvline(in_bench.median(), color='blue', linestyle='--', alpha=0.7)
    
    ax.set_title(col)
    if i == 0:
        ax.legend(fontsize=8)

for i in range(len(activity_features), 8):
    axes[i].set_visible(False)

plt.suptitle('Features de Actividad: Benchmark vs Resto', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: start_idx vs duration_ratio
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Start vs Duration
ax1 = axes[0]
not_bench = algo_features[~algo_features['in_benchmark']]
bench = algo_features[algo_features['in_benchmark']]

ax1.scatter(not_bench['start_idx'], not_bench['duration_ratio'], 
           c='lightgray', s=10, alpha=0.3, label=f'No benchmark ({len(not_bench)})')
ax1.scatter(bench['start_idx'], bench['duration_ratio'], 
           c='blue', s=30, alpha=0.7, label=f'Benchmark ({len(bench)})')

ax1.set_xlabel('Start Index (0=inicio, 1=fin del estudio)')
ax1.set_ylabel('Duration Ratio (proporción del estudio)')
ax1.set_title('¿Cuándo empiezan y cuánto duran?')
ax1.legend()

# Plot 2: Duration vs Active Ratio
ax2 = axes[1]
ax2.scatter(not_bench['duration_ratio'], not_bench['active_ratio'], 
           c='lightgray', s=10, alpha=0.3)
ax2.scatter(bench['duration_ratio'], bench['active_ratio'], 
           c='blue', s=30, alpha=0.7)

ax2.set_xlabel('Duration Ratio')
ax2.set_ylabel('Active Ratio (% tiempo activo sobre duración)')
ax2.set_title('¿Son continuos o tienen gaps?')

plt.tight_layout()
plt.show()

# Interpretación
print("\nInterpretación:")
print(f"  - Algoritmos del benchmark: start_idx medio = {bench['start_idx'].mean():.2f}")
print(f"  - Algoritmos del benchmark: duration_ratio medio = {bench['duration_ratio'].mean():.2f}")
print(f"  - Algoritmos fuera: start_idx medio = {not_bench['start_idx'].mean():.2f}")
print(f"  - Algoritmos fuera: duration_ratio medio = {not_bench['duration_ratio'].mean():.2f}")

---
## 4. Análisis de Features de Transición
---

¿Cómo evoluciona el rendimiento con la edad del algoritmo?

In [ ]:
# Visualizar features de transición
transition_features = [f for f in TRANSITION_FEATURES if f in algo_features.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(transition_features[:8]):
    ax = axes[i]
    
    in_bench = algo_features[algo_features['in_benchmark']][col]
    not_in_bench = algo_features[~algo_features['in_benchmark']][col]
    
    # Eliminar outliers extremos para visualización
    q_low, q_high = not_in_bench.quantile([0.01, 0.99])
    filtered_not = not_in_bench[(not_in_bench >= q_low) & (not_in_bench <= q_high)]
    filtered_in = in_bench[(in_bench >= q_low) & (in_bench <= q_high)]
    
    ax.hist(filtered_not, bins=30, alpha=0.5, color='gray', label='No benchmark', density=True)
    ax.hist(filtered_in, bins=30, alpha=0.7, color='blue', label='Benchmark', density=True)
    
    ax.set_title(col)
    if i == 0:
        ax.legend(fontsize=8)

for i in range(len(transition_features), 8):
    axes[i].set_visible(False)

plt.suptitle('Features de Transición: ¿Cómo evolucionan con la edad?', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Análisis de degradación: ¿Los algoritmos mejoran o empeoran?
print("\n" + "="*60)
print("ANÁLISIS DE DEGRADACIÓN")
print("="*60)

# Clasificar algoritmos por degradación
if 'return_decay' in algo_features.columns:
    improving = (algo_features['return_decay'] > 0.05).sum()
    stable = ((algo_features['return_decay'] >= -0.05) & (algo_features['return_decay'] <= 0.05)).sum()
    degrading = (algo_features['return_decay'] < -0.05).sum()
    
    print(f"\nDistribución de degradación (return_decay):")
    print(f"  - Mejorando (>5%): {improving} ({improving/len(algo_features)*100:.1f}%)")
    print(f"  - Estable (±5%): {stable} ({stable/len(algo_features)*100:.1f}%)")
    print(f"  - Degradando (<-5%): {degrading} ({degrading/len(algo_features)*100:.1f}%)")
    
    # Comparar benchmark vs no benchmark
    bench_decay = algo_features[algo_features['in_benchmark']]['return_decay'].mean()
    non_bench_decay = algo_features[~algo_features['in_benchmark']]['return_decay'].mean()
    
    print(f"\n  Return decay medio (benchmark): {bench_decay:.4f}")
    print(f"  Return decay medio (no benchmark): {non_bench_decay:.4f}")

# Tiempo hasta primer drawdown
if 'time_to_first_dd' in algo_features.columns:
    print(f"\nTiempo hasta primer drawdown relevante (>5%):")
    print(f"  - Benchmark: {algo_features[algo_features['in_benchmark']]['time_to_first_dd'].mean():.2f}")
    print(f"  - No benchmark: {algo_features[~algo_features['in_benchmark']]['time_to_first_dd'].mean():.2f}")
    print(f"  (1.0 = nunca tuvo DD relevante durante su vida)")

---
## 5. Test Estadístico: Benchmark vs No-Benchmark
---

In [ ]:
# Test de Mann-Whitney U para todas las features
print("="*80)
print("TEST DE MANN-WHITNEY U: ¿Qué distingue al benchmark?")
print("="*80)

all_test_features = (
    ACTIVITY_FEATURES + PERFORMANCE_FEATURES + 
    TRANSITION_FEATURES + BENCHMARK_FEATURES
)
all_test_features = [f for f in all_test_features if f in algo_features.columns]

results = []
for col in all_test_features:
    in_bench = algo_features[algo_features['in_benchmark']][col].dropna()
    not_in_bench = algo_features[~algo_features['in_benchmark']][col].dropna()
    
    if len(in_bench) > 5 and len(not_in_bench) > 5:
        stat, p = stats.mannwhitneyu(in_bench, not_in_bench, alternative='two-sided')
        
        diff = in_bench.median() - not_in_bench.median()
        direction = "↑" if diff > 0 else "↓"
        
        results.append({
            'feature': col,
            'p_value': p,
            'significant': p < 0.05,
            'bench_median': in_bench.median(),
            'non_bench_median': not_in_bench.median(),
            'diff': diff,
            'direction': direction,
        })

results_df = pd.DataFrame(results).sort_values('p_value')

print("\nFeatures más significativas (p<0.05):")
sig_results = results_df[results_df['significant']]
display(sig_results.style.format({
    'p_value': '{:.2e}',
    'bench_median': '{:.4f}',
    'non_bench_median': '{:.4f}',
    'diff': '{:+.4f}',
}))

print(f"\nTotal features significativas: {len(sig_results)} / {len(results)}")

---
## 6. Clustering de Dos Capas
---

**Capa 1: Life Profile** - Agrupar por patrón de actividad
**Capa 2: Financial Behavior** - Dentro de cada grupo, agrupar por rendimiento

In [ ]:
# Definir features para cada capa
LIFE_FEATURES = [
    'start_idx', 'end_idx', 'duration_ratio', 'active_ratio', 'n_gaps'
]

BEHAVIOR_FEATURES = [
    'sharpe', 'sortino', 'max_dd', 'ann_vol',
    'return_decay', 'sharpe_stability', 'time_to_first_dd',
    'corr_benchmark', 'hit_ratio'
]

# Filtrar features disponibles
life_features = [f for f in LIFE_FEATURES if f in algo_features.columns]
behavior_features = [f for f in BEHAVIOR_FEATURES if f in algo_features.columns]

print("Features para clustering:")
print(f"  Capa 1 (Life Profile): {life_features}")
print(f"  Capa 2 (Financial Behavior): {behavior_features}")

In [ ]:
# Ejecutar clustering de dos capas
two_layer_result = AlgoClusterer.two_layer_clustering(
    algo_features,
    life_features=life_features,
    behavior_features=behavior_features,
    life_method=ClusterMethod.HDBSCAN,
    behavior_method=ClusterMethod.GMM,
    n_life_clusters=4,
    n_behavior_clusters=3,
    min_cluster_size_for_subclustering=100,
    scaler_type=ScalerType.ROBUST,
)

In [ ]:
# Resumen de resultados
print("\n" + "="*80)
print("RESUMEN DE CLUSTERING DE DOS CAPAS")
print("="*80)

print(f"\nCapa 1 (Life Profile):")
print(f"  Método: {two_layer_result.life_profile_result.method.value}")
print(f"  Clusters: {two_layer_result.life_profile_result.n_clusters}")
print(f"  Silhouette: {two_layer_result.life_profile_result.silhouette:.3f}")
print(f"  Ruido: {two_layer_result.life_profile_result.n_noise}")

print(f"\nNombres de Life Profiles:")
for cluster_id, name in sorted(two_layer_result.life_profile_names.items()):
    size = two_layer_result.life_profile_result.cluster_sizes.get(cluster_id, 0)
    print(f"  {cluster_id}: {name} ({size} algos)")

print(f"\nCapa 2 (Financial Behavior):")
for life_id, beh_result in two_layer_result.behavior_results.items():
    life_name = two_layer_result.life_profile_names.get(life_id, f"L{life_id}")
    if beh_result is not None:
        print(f"  {life_name}: {beh_result.n_clusters} sub-clusters, silhouette={beh_result.silhouette:.3f}")
    else:
        print(f"  {life_name}: cluster único (muy pocos algos)")

print(f"\nTotal clusters combinados: {two_layer_result.n_total_clusters}")

In [ ]:
# Añadir labels al dataframe
algo_features['life_cluster'] = two_layer_result.life_profile_result.labels
algo_features['life_cluster_name'] = algo_features['life_cluster'].map(
    two_layer_result.life_profile_names
)
algo_features['combined_cluster'] = two_layer_result.combined_labels

# Distribución de clusters combinados
print("\nDistribución de clusters combinados:")
cluster_dist = algo_features['combined_cluster'].value_counts()
print(cluster_dist)

In [ ]:
# Visualizar Capa 1 con PCA
X_life = algo_features[life_features].copy()
X_life = X_life.replace([np.inf, -np.inf], np.nan).fillna(X_life.median())

scaler = RobustScaler()
X_life_scaled = scaler.fit_transform(X_life)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_life_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Plot 1: Life clusters
ax1 = axes[0]
labels = two_layer_result.life_profile_result.labels
unique_labels = sorted(set(labels))
colors = plt.cm.tab10(np.linspace(0, 1, max(10, len(unique_labels))))

for i, label in enumerate(unique_labels):
    mask = labels == label
    if label == -1:
        ax1.scatter(X_pca[mask, 0], X_pca[mask, 1], 
                   c='lightgray', s=10, alpha=0.3, label='Noise')
    else:
        name = two_layer_result.life_profile_names.get(label, f"C{label}")
        ax1.scatter(X_pca[mask, 0], X_pca[mask, 1], 
                   c=[colors[label % 10]], s=20, alpha=0.6, label=name)

ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')
ax1.set_title('Capa 1: Life Profile Clusters')
ax1.legend(loc='upper right', fontsize=8)

# Plot 2: Benchmark overlay
ax2 = axes[1]
not_bench_mask = ~algo_features['in_benchmark'].values
bench_mask = algo_features['in_benchmark'].values

ax2.scatter(X_pca[not_bench_mask, 0], X_pca[not_bench_mask, 1], 
           c='lightgray', s=10, alpha=0.3, label='No benchmark')
ax2.scatter(X_pca[bench_mask, 0], X_pca[bench_mask, 1], 
           c='blue', s=30, alpha=0.7, label='Benchmark')

ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
ax2.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')
ax2.set_title('Life Profile: Benchmark Overlay')
ax2.legend()

plt.suptitle('Clustering Capa 1: Perfil de Vida', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Perfil medio de cada Life Cluster
print("\n" + "="*60)
print("PERFIL MEDIO DE CADA LIFE CLUSTER")
print("="*60)

# Filtrar ruido
non_noise = algo_features[algo_features['life_cluster'] >= 0]

life_profiles = non_noise.groupby('life_cluster_name')[life_features + ['sharpe', 'max_dd']].mean()
display(life_profiles.style.format('{:.3f}').background_gradient(cmap='RdYlGn', axis=0))

---
## 7. Análisis de Representación por Life Cluster
---

In [ ]:
# Distribución del benchmark por life cluster
print("="*80)
print("DISTRIBUCIÓN DEL BENCHMARK POR LIFE CLUSTER")
print("="*80)

life_dist = non_noise.groupby('life_cluster_name').agg({
    'in_benchmark': ['sum', 'count'],
}).droplevel(0, axis=1)
life_dist.columns = ['n_benchmark', 'n_total']
life_dist['n_non_benchmark'] = life_dist['n_total'] - life_dist['n_benchmark']
life_dist['pct_benchmark'] = life_dist['n_benchmark'] / life_dist['n_total'] * 100
life_dist['pct_of_benchmark'] = life_dist['n_benchmark'] / life_dist['n_benchmark'].sum() * 100
life_dist['pct_of_universe'] = life_dist['n_total'] / life_dist['n_total'].sum() * 100
life_dist['representation_ratio'] = life_dist['pct_of_benchmark'] / life_dist['pct_of_universe']

display(life_dist.style.format({
    'pct_benchmark': '{:.1f}%',
    'pct_of_benchmark': '{:.1f}%',
    'pct_of_universe': '{:.1f}%',
    'representation_ratio': '{:.2f}x',
}).background_gradient(subset=['representation_ratio'], cmap='RdYlGn', vmin=0.5, vmax=2))

In [ ]:
# Visualizar representación
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Composición de cada cluster
ax1 = axes[0]
life_dist[['n_benchmark', 'n_non_benchmark']].plot(
    kind='bar', stacked=True, ax=ax1, color=['blue', 'lightgray']
)
ax1.set_title('Composición de cada Life Cluster')
ax1.set_xlabel('Life Cluster')
ax1.set_ylabel('Número de Algoritmos')
ax1.tick_params(axis='x', rotation=45)
ax1.legend(['Benchmark', 'No Benchmark'])

# Plot 2: Representation ratio
ax2 = axes[1]
colors = ['green' if r > 1 else 'red' for r in life_dist['representation_ratio']]
life_dist['representation_ratio'].plot(kind='bar', ax=ax2, color=colors, alpha=0.7)
ax2.axhline(1.0, color='black', linestyle='--', label='Neutral (1.0x)')
ax2.set_title('Ratio de Representación (Benchmark/Universo)')
ax2.set_xlabel('Life Cluster')
ax2.set_ylabel('Ratio')
ax2.tick_params(axis='x', rotation=45)
ax2.legend()

plt.tight_layout()
plt.show()

# Interpretación
print("\nInterpretación:")
for cluster_name in life_dist.index:
    ratio = life_dist.loc[cluster_name, 'representation_ratio']
    if ratio > 1.2:
        print(f"  ✓ Benchmark FAVORECE '{cluster_name}' ({ratio:.1f}x más que universo)")
    elif ratio < 0.8:
        print(f"  ✗ Benchmark EVITA '{cluster_name}' ({ratio:.1f}x menos que universo)")
    else:
        print(f"  ○ '{cluster_name}' representado proporcionalmente ({ratio:.1f}x)")

---
## 8. Comparación con Clustering Simple (Single-Layer)
---

Comparamos el enfoque de dos capas con clustering tradicional.

In [ ]:
# Features combinadas para clustering simple
all_clustering_features = life_features + behavior_features

print("="*80)
print("COMPARACIÓN: SINGLE-LAYER vs TWO-LAYER CLUSTERING")
print("="*80)

# Clustering simple con múltiples métodos
print("\nClustering simple (todas las features juntas):")
single_layer_results = AlgoClusterer.compare_methods(
    algo_features,
    n_clusters=5,
    features=all_clustering_features,
    scaler_type=ScalerType.ROBUST,
)

# Crear tabla comparativa
comparison_data = []
for method, result in single_layer_results.items():
    comparison_data.append({
        'Método': f"Single-{method.value.upper()}",
        'Clusters': result.n_clusters,
        'Ruido': result.n_noise,
        'Silhouette': result.silhouette,
    })

# Añadir two-layer
comparison_data.append({
    'Método': 'Two-Layer (HDBSCAN+GMM)',
    'Clusters': two_layer_result.n_total_clusters,
    'Ruido': two_layer_result.life_profile_result.n_noise,
    'Silhouette': two_layer_result.overall_silhouette,
})

comparison_df = pd.DataFrame(comparison_data).set_index('Método')
display(comparison_df.style.format({'Silhouette': '{:.3f}'}).background_gradient(
    subset=['Silhouette'], cmap='RdYlGn'
))

In [ ]:
# Seleccionar mejor modelo single-layer para comparación visual
best_single_method = max(single_layer_results.keys(), 
                         key=lambda m: single_layer_results[m].silhouette)
best_single_result = single_layer_results[best_single_method]

print(f"\nMejor método single-layer: {best_single_method.value.upper()}")
print(f"  Silhouette: {best_single_result.silhouette:.3f}")

# Guardar labels para comparación
algo_features['single_layer_cluster'] = best_single_result.labels

# Nombrar clusters single-layer
single_layer_names = name_clusters(
    algo_features, best_single_result.labels, all_clustering_features
)
algo_features['single_layer_name'] = algo_features['single_layer_cluster'].map(single_layer_names)

---
## 9. Guardar Resultados
---

In [ ]:
# Guardar features completas
algo_features.to_parquet(str(dp.algorithms.features.parent / "algo_features_enhanced.parquet"))
print("Guardado: data/processed/algo_features_enhanced.parquet")

# Guardar clusters
cluster_cols = [
    'in_benchmark', 
    'life_cluster', 'life_cluster_name',
    'combined_cluster',
    'single_layer_cluster', 'single_layer_name'
]
algo_features[cluster_cols].to_parquet(str(dp.processed.root / "algo_clusters_twolayer.parquet"))
print("Guardado: data/processed/algo_clusters_twolayer.parquet")

# Guardar comparación de métodos
comparison_df.to_csv(str(dp.processed.root / "clustering_methods_comparison.csv"))
print("Guardado: data/processed/clustering_methods_comparison.csv")

In [ ]:
# Resumen final
n_total = len(algo_features)
n_benchmark = algo_features['in_benchmark'].sum()
n_noise = (algo_features['life_cluster'] == -1).sum()

print("""
╔══════════════════════════════════════════════════════════════════════════╗
║         RESUMEN: ANÁLISIS Y CLUSTERING DE DOS CAPAS                      ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  DATOS ANALIZADOS                                                        ║
║  ────────────────                                                        ║""")
print(f"║    Total algoritmos: {n_total:<55}║")
print(f"║    En benchmark: {n_benchmark} ({n_benchmark/n_total*100:.1f}%){' '*(47-len(str(n_benchmark))-5)}║")
print(f"║    Total features: {len(algo_features.columns):<53}║")
print("║                                                                          ║")
print("║  FEATURES MEJORADAS                                                      ║")
print("║  ──────────────────                                                      ║")
print(f"║    Actividad: {len([f for f in ACTIVITY_FEATURES if f in algo_features.columns])} features (start, duration, gaps)              ║")
print(f"║    Rendimiento: {len([f for f in PERFORMANCE_FEATURES if f in algo_features.columns])} features (sharpe, dd, ulcer)             ║")
print(f"║    Transición: {len([f for f in TRANSITION_FEATURES if f in algo_features.columns])} features (decay, stability)              ║")
print(f"║    Benchmark: {len([f for f in BENCHMARK_FEATURES if f in algo_features.columns])} features (corr, beta, alpha)               ║")
print("║                                                                          ║")
print("║  CLUSTERING DE DOS CAPAS                                                 ║")
print("║  ─────────────────────────                                               ║")
print(f"║    Capa 1 (Life Profile): {two_layer_result.life_profile_result.n_clusters} clusters{' '*39}║")
print(f"║    Capa 2 (Behavior): sub-clusters por grupo de vida{' '*19}║")
print(f"║    Total combinados: {two_layer_result.n_total_clusters} clusters{' '*43}║")
print(f"║    Ruido: {n_noise} algoritmos{' '*52}║")
print("║                                                                          ║")
print("║  HALLAZGOS CLAVE                                                         ║")
print("║  ───────────────                                                         ║")

for cluster_name in life_dist.index:
    ratio = life_dist.loc[cluster_name, 'representation_ratio']
    if ratio > 1.2:
        msg = f"    Benchmark FAVORECE '{cluster_name}': {ratio:.1f}x"
        print(f"║{msg:<73}║")
    elif ratio < 0.8:
        msg = f"    Benchmark EVITA '{cluster_name}': {ratio:.1f}x"
        print(f"║{msg:<73}║")

print("║                                                                          ║")
print("╚══════════════════════════════════════════════════════════════════════════╝")

print("\n\nArchivos generados:")
print("  - data/processed/algo_features_enhanced.parquet (features completas)")
print("  - data/processed/algo_clusters_twolayer.parquet (clusters de dos capas)")
print("  - data/processed/clustering_methods_comparison.csv")

print("\nSiguiente paso: 04_regime_analysis.ipynb para análisis de regímenes")